# FineBERT

In [ ]:
import torch
import transformers

if torch.cuda.is_available():
    print( {torch.cuda.get_device_name(0)})

### Pipeline Finebert model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

print(model)
print(model.config.id2label)
print(tokenizer.vocab_size)



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
model.eval()

In [ ]:
import pandas as pd

train_df = pd.read_csv('../data/processed/train.csv')
valid_df = pd.read_csv('../data/processed/valid.csv')
test_df = pd.read_csv('../data/processed/test.csv')

In [ ]:
X_test_raw = test_df['sentence']
X_test = test_df['clean_text']
y_test = test_df['sentiment']

In [ ]:
from transformers import pipeline

finbert = pipeline(
    "sentiment-analysis",
    model = "ProsusAI/finbert",
    tokenizer = "ProsusAI/finbert"

)

In [ ]:
zero_shot_results = finbert(list(X_test_raw), batch_size=32)
y_pred = [r['label'].lower() for r in zero_shot_results]

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

### Manual model

In [65]:
train_df = pd.read_csv('../data/processed/train.csv')
valid_df = pd.read_csv('../data/processed/valid.csv')
test_df = pd.read_csv('../data/processed/test.csv')

In [66]:
X_train = train_df['clean_text']
y_train = train_df['sentiment']

X_val = valid_df['clean_text']
y_val = valid_df['sentiment']

X_test = test_df['clean_text']
y_test = test_df['sentiment']

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)
y_test = le.transform(y_test)

In [ ]:
class CustomDataset(Dataset):

    def __init__(self, text, labels, tokenizer, max_len = 128):
        self.text = text    
        self.labels = labels
        self.tokenizer = tokenizer
    
    def __len__(self):
        return len(self.text)
    
    def __getitem__(self, idx):
        token = tokenizer.tokenize(self.text[idx])
        label = self.labels[idx]
        return token, label

In [64]:
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,